# This file contains the Data Clean Script fot the project.

## For the Gaint Dataset intensity.csv

In [12]:
# Constants
RAW_FILE_PATH = "data/raw/intensities.csv"
CLEAN_FILE_PATH = "data/clean/intensities_t1.csv"
DEM_FILE_PATH = "mi30_grd/mi30_grd/hdr.adf"

In [13]:

import pandas as pd

df = pd.read_csv(RAW_FILE_PATH)

/tmp/ipykernel_182001/3840777028.py:3: DtypeWarning: Columns (4: CHIMP_eventcode, 19: CHIMP_Magnitude_Type) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_FILE_PATH)



If you encounter a `FileNotFoundError`, the dataset has not been extracted yet.

The repository does not include `intensities.csv` directly because the file is too large for practical version control. Instead, it is stored as a compressed archive.

To resolve the issue:

1. Navigate to the `root/archive/LargeFiles` directory.
2. Extract `EarthQuakeDataset.zip`.
3. Copy `intensities.csv` into the `data/raw` directory.
4. Rerun the notebook or script.

Expected location:

```text
data/raw/intensities.csv
```

The dataset is distributed as a ZIP archive to reduce repository size while still allowing the full dataset to be included with the project.


In [14]:
import numpy as np
import pandas as pd
import rasterio


def compute_surface_depth(
    df: pd.DataFrame,
    dem_filepath: str,
    lon_col: str = "Longitude",
    lat_col: str = "Latitude",
    depth_col: str = "Depth",
    elevation_col: str = "elevation",
    depth_surface_col: str = "depth_surface",
) -> pd.DataFrame:
    """Samples elevation from a local DEM raster and calculates depth below ground surface.

    Parameters
    ----------
    df : pd.DataFrame
        Input earthquake DataFrame.
    dem_filepath : str
        Path to the DEM file (GeoTIFF or ArcGrid header folder).
    lon_col : str, optional
        Column name for longitude coordinates, by default "Longitude".
    lat_col : str, optional
        Column name for latitude coordinates, by default "Latitude".
    depth_col : str, optional
        Column name for USGS depth in km, by default "Depth".

    Returns
    -------
    pd.DataFrame
        The modified DataFrame containing 'elevation' and
        'depth_surface'.
    """
    # 1. Zip coordinate pairs (Longitude, Latitude)
    coords = list(zip(df[lon_col], df[lat_col]))

    # 2. Batch-sample DEM raster using memory-mapped C-speed iterator
    with rasterio.open(dem_filepath) as src:
        elevations_m = np.fromiter(
            (val[0] for val in src.sample(coords)), dtype=np.float32
        )

    # 3. Vectorized unit conversion (m to km) and depth computation
    df[elevation_col] = elevations_m / 1000.0
    df[depth_surface_col] = (df[elevation_col] + df[depth_col]).clip(
        lower=0.0
    ).round(decimals=5)
    return df

In [15]:
dataset_to_feature_dict = {
	"USGS_ComCatID": "usgs_id",
	"Longitude": "longitude",
	"Latitude": "latitude",
	"Preferred_Magnitude": "magnitude",
	"Depth": "depth_from_sea",
	"Rhypo": "rhypo",
	"Regime": "regime",
	"Intensity":"mmi",
}

dataset_filter_keys = dataset_to_feature_dict.keys()
df_filtered = df[dataset_filter_keys]
df_filtered.rename(columns=dataset_to_feature_dict, inplace=True)

# as the null column were only 5% of 695k total rows, we can drop the null values and 
# save the cleaned dataset to a new file
df_dropped = df_filtered.dropna()

df_dropped = compute_surface_depth(
	df_dropped, DEM_FILE_PATH, lon_col="longitude", 
	lat_col="latitude", depth_col="depth_from_sea", 
	elevation_col="elevation", depth_surface_col="depth_surface")


df_dropped.to_csv(CLEAN_FILE_PATH, index=False)

If you encounter a `FileNotFoundError`, the DEM dataset has not been extracted yet.

The repository does not include the uncompressed DEM files directly because they are too large for practical version control. Instead, the DEM is stored as a compressed archive in the project directory.

To resolve the issue:

1. Locate the compressed DEM archive (`mi30_grd.zip`) in the project_root/arhieve/LargeFiles
2. Extract `mi30_grd.zip` directly into the current directory.
3. Ensure the extracted folder structure creates `mi30_grd/mi30_grd/hdr.adf`.
4. Rerun the notebook or script.

Expected location:

```text
mi30_grd/mi30_grd/hdr.adf

```

The DEM is distributed as a ZIP archive to reduce repository size while still allowing the full dataset to be included with the project.